In [1]:
import json
import os
from tqdm import tqdm

In [2]:
def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)

def save_json(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=4)

In [3]:
total_results = {}

In [4]:
import re

def get_number_from_string(text):
    # Tìm dấu gạch dưới và các chữ số ở cuối chuỗi
    match = re.search(r'_(\d+)$', text)
    if match:
        return int(match.group(1)) # Trả về số dưới dạng string
    return None

In [5]:
CACHE_URL_PATH = r"data\cache\mapping\hash2url.json"
CACHE_SEARCH_PATH = r"data\cache\mapping\hash_id_IR_20.json"
METADATA_PATH = r"data\metadata_9000q.json"

DIR_SEARCH = "data/cache/IR_20"
DIR_WIKI = "data/cache/wiki_pages_tool_1"

In [6]:
def flatten_wiki_cache(file_path):
    # 1. Load dữ liệu từ file
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
    except Exception as e:
        print(f"Lỗi khi đọc file: {e}")
        return {}

    final_cache = {}

    for main_id, tasks in raw_data.items():
        for task_name, items in tasks.items():
            for hash_id, url in items.items():
                final_cache[hash_id] = url

    return final_cache

FILE_PATH = r'data\cache\all_hash_id_crawl_wiki_embed.json' 
url_mapping = flatten_wiki_cache(FILE_PATH)

print(f"Tổng số HashID thu được: {len(url_mapping)}")

Tổng số HashID thu được: 4074


In [7]:
CACHE_URL_WIKI = load_json(CACHE_URL_PATH)
CACHE_SEARCH = load_json(CACHE_SEARCH_PATH)
metadata = load_json(METADATA_PATH)
    
CACHE_URL_WIKI.update(url_mapping)

# SEARCH_MAPPING = load_json(r'data\cache\mapping\hash_id copy.json')
# CACHE_SEARCH.update(SEARCH_MAPPING)
# CACHE_SEARCH.update(load_json(r'data\cache\mapping\hash_id.json'))

# save_json(CACHE_SEARCH, CACHE_SEARCH_PATH)

In [8]:
len(CACHE_URL_WIKI)

53017

In [9]:
import re

def load_retrieval_map(data_input):
    """
    Map ID (WIKI_10, CRAWL_NER_WIKI_2,...) trực tiếp với hash_id theo thứ tự 1-1.
    """
    result_map = {}
    
    for key, entry in data_input.items():
        crawl_docs = entry.get("crawl_doc", [])
        hash_ids = entry.get("hash_id", [])
        for i, doc_text in enumerate(crawl_docs):
            for c in range(1, 50):
                s = "CRAWL_WIKI_{c}"
                if f"[{s}]" in doc_text:
                    result_map[s] = hash_ids[i]
                    break
                    
                s_2 = f"CRAWL_NER_WIKI_{c}" #CRAWL_NER_WIKI_8
                
                if f"[{s_2}]" in doc_text:
                    result_map[s_2] = hash_ids[i]
                    break
           
    return result_map

In [10]:
def extract_detail_regex(text):
    pattern = r'(?i)Detailed explanation[:\s]*(.*)'
    match = re.search(pattern, text)
    
    if match:
        return match.group(1).strip()
    return ""

In [11]:
def get_GG2URL(search_kw, search_kw_raw):
    dict_gg_id = {}
    if search_kw in CACHE_SEARCH or search_kw_raw in CACHE_SEARCH:
        hash_id_gg = CACHE_SEARCH.get(search_kw) or CACHE_SEARCH.get(search_kw_raw)
        if '.json' not in hash_id_gg:
            hash_id_gg = f"{hash_id_gg}.json"
        file_gg = f'{DIR_SEARCH}/{hash_id_gg}'
        gg_data = load_json(file_gg)['google']['search_results']
        
        for idx, item in enumerate(gg_data, 1):
            dict_gg_id[f"GG_{idx}"] = item['link']
        
    return dict_gg_id        

In [12]:
def extract_detail_regex(text):
    pattern = r'(?i)Detailed explanation[:\s]*'
    match = re.search(pattern, text)
    
    if match:
        return text[match.start():]  # Lấy từ vị trí bắt đầu "Detailed explanation" đến cuối
    return text  # Trả về text gốc nếu không tìm thấy

In [ ]:
text = "Reference: [GG_7], [CRAWL_WIKI_1]. Detailed explanation: The provided context describes that myeloproliferative neoplasms are frequently associated with somatic mutations, specifically mentioning the JAK2 mutation. In particular, the context indicates that the constitutive activation of the JAK-STAT pathway, which is observed through the JAK2 V617F mutation, is a key driver mutation in these diseases.Reference: [GG_6], [CRAWL_WIKI_1]. Detailed explanation: The provided context states that myeloproliferative neoplasms are often associated with somatic mutations, for example in the JAK2 gene. In polycythemia vera specifically, it is noted that the JAK2 V617F mutation is present in greater than 95% of cases, making it a key mutation associated with these diseases.Reference: [GG_8], [CRAWL_NER_WIKI_5]. Detailed explanation: The context states that the development of myeloproliferative neoplasms is driven by mutations in genes including JAK2, CALR, and MPL, with a specific mention that in essential thrombocythemia—a type of myeloproliferative neoplasm—a mutation in the JAK2 kinase (specifically the V617F mutation) is present in 40–50% of cases and is diagnostic if present.Reference: [CRAWL_NER_WIKI_5], [CRAWL_WIKI_1]. Detailed explanation: The context indicates that in essential thrombocythemia—a type of myeloproliferative neoplasm—a mutation in the JAK2 gene, specifically the JAK2 V617F mutation, is commonly found and is used diagnostically in about 40–50% of cases. Additionally, while CALR and MPL mutations are also mentioned, the JAK2 V617F mutation is the primary gene mutation associated with the disease."

extract_detail_regex(text)



'Detailed explanation: The provided context describes that myeloproliferative neoplasms are frequently associated with somatic mutations, specifically mentioning the JAK2 mutation. In particular, the context indicates that the constitutive activation of the JAK-STAT pathway, which is observed through the JAK2 V617F mutation, is a key driver mutation in these diseases.Reference: [GG_6], [CRAWL_WIKI_1]. Detailed explanation: The provided context states that myeloproliferative neoplasms are often associated with somatic mutations, for example in the JAK2 gene. In polycythemia vera specifically, it is noted that the JAK2 V617F mutation is present in greater than 95% of cases, making it a key mutation associated with these diseases.Reference: [GG_8], [CRAWL_NER_WIKI_5]. Detailed explanation: The context states that the development of myeloproliferative neoplasms is driven by mutations in genes including JAK2, CALR, and MPL, with a specific mention that in essential thrombocythemia—a type of

# DIRECT

In [13]:
BASE_9000 = "DIRECT_9000_medhop_results/all_ans"
DIR_SEARCH = "data/cache/IR_20"
DIR_WIKI = "data/cache/wiki_pages_tool_1"

from tqdm import tqdm

In [14]:
output_direct = {}

for qidx in tqdm(list(metadata.keys())):
    item = metadata[qidx]
    
    if item['type_q'] != 'direct':
        continue
    
    output_direct[qidx] = {
        'QIDX': qidx        
    }
    
    file_path = f"{BASE_9000}/R4_step_3b_all_ans_{qidx}.json"
    tmp_data = load_json(file_path)
    
    final_answer = tmp_data['1']['0']['0']['final_answer']
    question = tmp_data['1']['0']['0']['question']
    output_direct[qidx]['Question'] = question
    search_kw = tmp_data['1']['0']['0']['search_keyword'].lower().strip()
    search_kw_raw = search_kw + " "
    short_answer = final_answer['short_answer']
    output_direct[qidx]['short_answer'] = short_answer
    long_answer = final_answer['long_answer']
    
    retrieval_information = tmp_data['1']['0']['0']['retrieval_information']
    
    if final_answer['selection_status'] != 'selected':
        list_references = "This answer, generated by a zero-shot LLM, is not explainable."
    else:
        sources = final_answer['sources']
    
    dict_gg_id2URL = get_GG2URL(search_kw, search_kw_raw)
    dict_crawl_wiki_data = load_retrieval_map(retrieval_information)
    dict_results_src = {}

    for src in sources:
        if src not in long_answer:
            continue
        if src in dict_gg_id2URL:
            dict_results_src[src] = dict_gg_id2URL[src]
            
        if src in dict_crawl_wiki_data:
            wiki_hashid = dict_crawl_wiki_data[src]
            dict_results_src[src] = CACHE_URL_WIKI.get(wiki_hashid, "")
            
    list_references = ""

    for src, link in dict_results_src.items():
        list_references += f"[{src}] {link}\n"    
        
    detail_exaplanations = extract_detail_regex(long_answer).replace("Detailed explanation", "\nExplanation")
    
    
    #CONCLUSION
    output_direct[qidx]['long_answer'] = f"""Question: {question}\nAnswer: {short_answer}\nDetail explanations: {detail_exaplanations}\nReferences:\n{list_references}"""
    
            

100%|██████████| 9000/9000 [01:10<00:00, 126.77it/s]


In [15]:
save_json(output_direct, 'data/EVIDENCE/reference_direct_9000.json')
print(len(output_direct.keys()))

7242


In [16]:
for qidx in output_direct:
    if qidx in total_results:
        print("ERROR")
    else:
        total_results[qidx] = output_direct[qidx]

# SEQUENCE

In [17]:
output_seq = {}
BASE_SEQ = "SEQUENCE_9000_medhop_results/all_ans"
seq_combine = load_json(r'SEQUENCE_9000_medhop_results\seq_combine_full.json')

for qidx in tqdm(list(metadata.keys())):    
    item = metadata[qidx]
    if item['type_q'] != 'sequence':
        continue
    
    output_seq[qidx] = {}    
    
    file_path = f"{BASE_SEQ}/R4_step_3b_all_ans_{qidx}.json"
    tmp_data = load_json(file_path)
    
    prev_answer = ""
    original_question = item['Question']
    output_seq[qidx]['QIDX'] = qidx
    output_seq[qidx]['Question'] = original_question
    
    output_seq[qidx]['long_answer'] = "Question: " + original_question + '\n\n' + "Follow the reasoning step by step:\n"
    
    for step, step_data in tmp_data['1']['0'].items():
        step = str(step)
        
        final_answer = tmp_data['1']['0'][step]['final_answer']
        question = tmp_data['1']['0'][step]['question']
        search_kw = tmp_data['1']['0'][step]['search_keyword']
                
        short_answer = final_answer['short_answer']
        long_answer = final_answer['long_answer']
        
        if step != '0':
            search_kw = search_kw + " " + prev_answer
            
        search_kw = search_kw.lower().strip()
        search_kw_raw = search_kw + " "
        prev_answer = short_answer.lower()
      
        if final_answer['selection_status'] != 'selected':
            list_references = "This answer, generated by a zero-shot LLM, is not explainable."
        else:
            sources = final_answer['sources']
        
        retrieval_information = tmp_data['1']['0'][step]['retrieval_information']
        dict_gg_id2URL = get_GG2URL(search_kw, search_kw_raw)
        dict_crawl_wiki_data = load_retrieval_map(retrieval_information)
        dict_results_src = {}

        for src in sources:
            if src not in long_answer:
                continue
            if src in dict_gg_id2URL:
                dict_results_src[src] = dict_gg_id2URL[src]
                
            if src in dict_crawl_wiki_data:
                wiki_hashid = dict_crawl_wiki_data[src]
                dict_results_src[src] = CACHE_URL_WIKI.get(wiki_hashid, "")
                
        list_references = ""

        for src, link in dict_results_src.items():
            list_references += f"[{src}] {link}\n"    
            
        detail_exaplanations = extract_detail_regex(long_answer).replace("Detailed explanation", "\nExplanation")
        
        
        output_seq[qidx]['long_answer'] += f"""Find answer for subquestion {int(step)+1}: {question}\nAnswer: {short_answer}\nDetail explanations: {detail_exaplanations}\nReferences:\n{list_references}\n"""
        
    #CONCLUSION
    conclusion_ans = seq_combine[qidx]
    output_seq[qidx]['long_answer'] += "\nConclusion:\n" + f"Answer: {conclusion_ans['short_answer']}\n" + f"Explanation: {conclusion_ans['long_answer']}"
    output_seq[qidx]['short_answer'] = conclusion_ans['short_answer']


  0%|          | 0/9000 [00:00<?, ?it/s]

100%|██████████| 9000/9000 [00:16<00:00, 529.53it/s]


In [18]:
print(len(output_seq.keys()))
save_json(output_seq, 'data/EVIDENCE/reference_sequence_9000.json')

1682


In [19]:
for qidx in output_seq:
    if qidx in total_results:
        print("ERROR")
    else:
        total_results[qidx] = output_seq[qidx]

# PARALLEL

In [20]:
output_parallel = {}
BASE_PAR = "PARALLEL_9000_medhop_results/all_ans"

for qidx in tqdm(list(metadata.keys())):    
    item = metadata[qidx]
       
    if item['type_q'] != 'parallel':
        continue
    output_parallel[qidx] = {'QIDX': qidx}
    
    file_path = f"{BASE_PAR}/R4_step_3b_all_ans_{qidx}.json"

    tmp_data = load_json(file_path)
    original_question = item['Question']
    output_parallel[qidx]['Question'] = original_question

    output_parallel[qidx]['long_answer'] = f"Question: {original_question}\n\nFollow the independent reasoning steps:\n"
    for qi in ["1", "2", "3", "4"]:
        if qi not in tmp_data:
            break      
        
        for step, step_data in tmp_data[qi]['0'].items():
            step = str(step)
            final_answer = tmp_data[qi]['0'][step]['final_answer']
            question = tmp_data[qi]['0'][step]['question']
            search_kw = tmp_data[qi]['0'][step]['search_keyword'].lower().strip()
            
            short_answer = final_answer['short_answer']
            long_answer = final_answer['long_answer']
           
            search_kw_raw = search_kw + " "            
            
            if final_answer['selection_status'] != 'selected':
                list_references = "This answer, generated by a zero-shot LLM, is not explainable."
            else:
                sources = final_answer['sources']
                
            retrieval_information = tmp_data[qi]['0'][step]['retrieval_information']
            dict_gg_id2URL = get_GG2URL(search_kw, search_kw_raw)
            dict_crawl_wiki_data = load_retrieval_map(retrieval_information)
            dict_results_src = {}

            for src in sources:
                if src not in long_answer:
                    continue
                if src in dict_gg_id2URL:
                    dict_results_src[src] = dict_gg_id2URL[src]
                    
                if src in dict_crawl_wiki_data:
                    wiki_hashid = dict_crawl_wiki_data[src]
                    dict_results_src[src] = CACHE_URL_WIKI.get(wiki_hashid, "")
                    
            list_references = ""

            for src, link in dict_results_src.items():
                list_references += f"[{src}] {link}\n"    
                
            detail_exaplanations = extract_detail_regex(long_answer).replace("Detailed explanation", "\nExplanation")

            output_parallel[qidx]['long_answer'] += f"""Find answer for question: {question}\nAnswer: {short_answer}\nDetail explanations: {detail_exaplanations}\nReferences:\n{list_references}\n"""
        
    #CONCLUSION
    conclusion_ans = tmp_data['final_answer']
    output_parallel[qidx]['long_answer'] += "\nConclusion:\n" + f"Answer: {conclusion_ans['short_answer']}\n" + f"Explanation: {conclusion_ans['long_answer']}"
    output_parallel[qidx]['short_answer'] = conclusion_ans['short_answer']

100%|██████████| 9000/9000 [00:03<00:00, 2928.85it/s]


In [21]:
save_json(output_parallel, 'data/EVIDENCE/reference_parallel_9000.json')

In [22]:
for qidx in output_parallel:
    if qidx in total_results:
        print("ERROR")
    else:
        total_results[qidx] = output_parallel[qidx]

In [23]:
import pandas as pd

data_list = []
for qidx, info in total_results.items():
    if 'Question' not in total_results[qidx]:
        print(qidx)
        
    data_list.append({
        'QIDX': qidx,
        'Question': info['Question'],
        'Short Answer': info['short_answer'],
        'Long Answer': info['long_answer']
    })

df = pd.DataFrame(data_list)

# Xuất ra CSV (sử dụng utf-8-sig để mở bằng Excel không bị lỗi font tiếng Việt)
df.to_csv('data/SUBMIT/submission_9000q.csv', index=False, encoding='utf-8-sig')

print("Đã lưu file CSV thành công!")

Đã lưu file CSV thành công!


In [24]:
import pandas as pd

# Đọc 2 file
df_a = pd.read_csv('data/SUBMIT/submission_1000q.csv')
df_b = pd.read_csv('data/SUBMIT/submission_9000q.csv')

# Nối hàng
df_combined = pd.concat([df_a, df_b], ignore_index=True)

# Lưu lại
df_combined.to_csv('submission.csv', index=False, encoding='utf-8-sig')